[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/05_result_interpretation/05_analysis_viz.ipynb)


# 05 — 결과 해석·시각화

> 본문 [`05_result_interpretation.md`](05_result_interpretation.md) 와 **한 절씩 짝지어** 보세요.
> 이 노트북의 표·그래프·수치는 **여러분이 직접 돌린 결과**(`my_run/`)에서 계산합니다.
> **① 직접 설계 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 갑니다. 설계 셀은 NVIDIA GPU 전용이에요(CPU 폴백 없음) — Colab 이면 **런타임 → 런타임 유형 변경 → T4 GPU**.

## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `05_result_interpretation/` 로 이동합니다. 로컬에서 `05_result_interpretation/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "05_result_interpretation"
PIP_PKGS = "pandas matplotlib"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 직접 돌려보기 — 내 결과 만들기

- 학습용 규모 `num_designs=8 --budget=4` (레퍼런스 결과는 num_designs=100)
- 소요 시간 실측 **585초**(최종 4개) — **24GB GPU · 가중치 캐시** 기준이에요. Colab T4 는 가속 커널이 꺼져 더 걸리고(T4 실측치 없음), 첫 실행은 가중치 ~6GB 다운로드가 더 붙어요.
- 건너뛰면 아래 분석이 커밋된 레퍼런스 결과로 이어집니다.

In [ ]:
SPEC, PROTOCOL = "example/vanilla_protein/1g13prot.yaml", "protein-anything"
NUM_DESIGNS, BUDGET = 8, 4

import shutil
OUT = MY.resolve()

def _gpu():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return shutil.which("nvidia-smi") is not None

if not _gpu():
    print("GPU 런타임이 아니라 설계 실행을 건너뜁니다 — 아래 분석은 레퍼런스 결과로 이어집니다.")
    print("  · 직접 돌리려면 Colab [런타임 → 런타임 유형 변경 → T4 GPU] 후 이 셀을 다시 실행하세요.")
else:
    SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세·타깃 CIF 가 들어 있는 BoltzGen 소스
    if not SRC.exists():
        _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')
    if not _have("boltzgen"):
        _run(f'"{sys.executable}" -m pip -q install -e "{SRC}"')
    try:
        _run(f'cd "{SRC}" && boltzgen run {SPEC} --output "{OUT}" --protocol {PROTOCOL} '
             f'--num_designs {NUM_DESIGNS} --budget {BUDGET}')
        print("\n내 결과 →", OUT / "final_ranked_designs")
    except Exception as e:
        print("\n설계 실행이 도중에 멈췄어요 —", type(e).__name__)
        print("  · 이 셀을 다시 실행하면 같은 --output 산출물을 재사용해 이어서 끝냅니다(실측 763초 → 재개 486초).")
        print("  · GPU 메모리가 부족했다면 NUM_DESIGNS 4, BUDGET 2 로 줄이세요.")

## 2) 결과 CSV 열기 — 컬럼이 몇 개인지부터 (본문 5.1)

Ch.04 에서 스모크 실행을 했다면 그 결과를 그대로 이어받습니다(05 에서 다시 안 돌려도 돼요).
한 디자인이 한 행, 컬럼은 수백 개예요. 이 많은 걸 다 볼 필요는 없고 실제로 자주 보는 건 10여 개뿐입니다.

In [ ]:
from boltzgen_viz import load_metrics, metrics_overview
import pandas as pd

inherit_run("../04_basic_usage/my_run")
CSV = str(find_one("final_designs_metrics_*.csv", "data/vanilla"))
df  = pd.read_csv(CSV)
print("최종 선별 디자인", len(df), "개 | 컬럼", df.shape[1], "개")

## 3) 자주 보는 메트릭 한 표로 (본문 5.2~5.4)

신뢰도(pTM·ipTM) · 위치오차(PAE) · 구조편차(RMSD 두 종류) · 인터페이스(H-bond·염다리·ΔSASA)를 나란히 놓습니다.
읽는 기준은 pTM 0.7 이상 양호, ipTM 0.5 이상이면 결합 가능성 양호, PAE 5Å 미만, `filter_rmsd` 2Å 미만 우수예요.

개발성(본문 5.5)·친화도(본문 5.6)·다양성/신규성(본문 5.7)은 프로토콜과 실행 옵션에 따라만 생성돼요 —
항체는 Ch.08, 소분자는 Ch.10 에서 다룹니다.

In [ ]:
view = df[cols_in(df, "id", "final_rank", "design_ptm", "design_to_target_iptm", "complex_plddt",
                  "min_design_to_target_pae", "filter_rmsd", "designfolding-filter_rmsd",
                  "plip_hbonds_refolded", "plip_saltbridge_refolded",
                  "delta_sasa_refolded")].sort_values("final_rank")
display(view)

if {"filter_rmsd", "designfolding-filter_rmsd"} <= set(df.columns):
    solo = df[(df["filter_rmsd"] < 2.0) & (df["designfolding-filter_rmsd"] >= 5.0)]
    print(f"복합체 RMSD 는 2A 미만인데 단독 재접힘 RMSD 가 5A 이상인 디자인 {len(solo)}/{len(df)}개")
    print("  타깃이 있어야만 그 모양이 유지된다는 신호예요 (본문 5.3).")

## 4) 표를 그림 한 장으로 (본문 5.10)

숫자 열 개 열을 눈으로 훑는 대신 2×2 개요로 봅니다.
pTM(보라)·ipTM(주황)·RMSD(청록) 바 + 길이↔H-bond 산점도, 임계선은 pTM 0.7 / ipTM 0.5 / RMSD 2.0Å.

> 내가 만든 그림은 `my_05_vanilla_metrics.png` 로 저장돼요(본문에 실린 레퍼런스 그림을 덮어쓰지 않도록).

In [ ]:
rows = load_metrics(CSV)
FIG  = my_fig("05_vanilla_metrics.png")
metrics_overview(rows, "Vanilla Protein — Design Metrics Overview", FIG)
from IPython.display import Image; Image(FIG)

## 5) 지표끼리 겹치나 — 상관관계 (본문 5.10)

그림에서 "구조는 대체로 괜찮은데 인터페이스에서 갈린다"가 보였다면, 그 인터페이스 지표들이
서로 같은 말을 반복하는 건 아닌지 확인할 차례예요. 상관이 낮을수록 각자 다른 정보를 준다는 뜻입니다.

> 직접 돌린 결과(4~8개)는 표본이 작아 상관계수가 불안정해요 — 방향만 보세요.

In [ ]:
m  = df[cols_in(df, "design_ptm", "design_to_target_iptm", "filter_rmsd",
                 "plip_hbonds_refolded", "plip_saltbridge_refolded",
                 "delta_sasa_refolded", "num_design")].astype(float)
cm = m.corr().round(2)

if {"design_ptm", "design_to_target_iptm"} <= set(cm.columns):
    print("pTM ↔ ipTM 상관", cm.loc["design_ptm", "design_to_target_iptm"],
          "— 0 에 가깝거나 음수면 서로 다른 걸 재고 있다는 뜻이라 둘을 함께 봐야 해요 (본문 5.2)")
if {"plip_hbonds_refolded", "delta_sasa_refolded"} <= set(cm.columns):
    print("H-bond ↔ ΔSASA 상관", cm.loc["plip_hbonds_refolded", "delta_sasa_refolded"],
          "— 클수록 두 지표가 같은 얘기를 반복한다는 뜻이에요")
print("\n판정 — 절댓값이 0.8 을 넘는 쌍은 둘 중 하나만 봐도 되고,")
print("0 근처인 쌍은 반드시 함께 봐야 해요. 그래서 순위도 한 지표로 매기지 않습니다.")
cm

## 6) 이 순위 컬럼들은 어디서 왔나 (본문 4.2·5.8)

표에 `final_rank` 가 있는데, 정작 메트릭을 계산한 **analysis 스텝**의 CSV 에는 없어요.
순위는 그 다음 **filtering 스텝**이 붙입니다. 두 CSV 의 컬럼을 차분하면 무엇이 덧붙었는지 그대로 보여요.

In [ ]:
POOL = find_one("all_designs_metrics.csv", "data/vanilla")
pool = pd.read_csv(POOL)

agg = None
try:
    agg = pd.read_csv(find_one("aggregate_metrics_analyze.csv", "data/vanilla"))
except Exception as e:
    print("aggregate_metrics_analyze.csv 를 못 찾았습니다 —", type(e).__name__)

print("\n선별 전 전체 풀", len(pool), "개 | 컬럼", pool.shape[1], "개")
if agg is not None:
    added = [col for col in pool.columns if col not in agg.columns]
    print("analysis 산출", agg.shape[1], "컬럼 → filtering 이", len(added), "개를 덧붙임")
    for kind, pre in [("하드필터 통과 여부", "pass_"), ("메트릭별 순위", "rank_"), ("종합", "")]:
        got = [a for a in added if a.startswith(pre)] if pre else \
              [a for a in added if a in ("final_rank", "max_rank", "secondary_rank", "quality_score")]
        if got:
            print(f"   {kind:16s}", ", ".join(got[:6]) + (" ..." if len(got) > 6 else ""))

try:
    per = pd.read_csv(find_one("per_target_metrics_analyze.csv", "data/vanilla"))
    print("\nper_target_metrics_analyze.csv — 타깃 1개가 1행인 요약", per.shape)
except Exception as e:
    print("\nper_target_metrics_analyze.csv 건너뜀 —", type(e).__name__)

## 7) 왜 pTM 1등이 최종 1등이 아닐까 (본문 5.8)

가장 자주 나오는 질문이에요. 답은 방금 본 `rank_*` 여섯 개에 있습니다.
BoltzGen 은 한 메트릭으로 줄세우지 않고, 여섯 지표를 각각 순위화한 뒤

```
rank_design_to_target_iptm · rank_design_ptm · rank_neg_min_design_to_target_pae
rank_plip_hbonds_refolded · rank_plip_saltbridge_refolded · rank_delta_sasa_refolded
        ↓
   max_rank → secondary_rank → final_rank
```

로 종합해요. 아래 표에서 각 디자인의 여섯 순위와 그 결과를 나란히 봅니다.

In [ ]:
RANKS = cols_in(df, "rank_design_to_target_iptm", "rank_design_ptm",
                "rank_neg_min_design_to_target_pae", "rank_plip_hbonds_refolded",
                "rank_plip_saltbridge_refolded", "rank_delta_sasa_refolded")
s = df.sort_values("final_rank")

if not RANKS:
    print("이 실행 CSV 에는 rank_* 컬럼이 없어 이 절은 건너뜁니다.")
else:
    tbl = s[cols_in(s, "final_rank", "id") + RANKS + cols_in(s, "max_rank", "secondary_rank")]
    display(tbl.rename(columns={col: col.replace("rank_", "") for col in RANKS}).set_index("final_rank"))

    if "max_rank" in s.columns:
        ok = int((s[RANKS].max(axis=1) == s["max_rank"]).sum())
        print(f"max_rank = 여섯 순위 중 가장 나쁜 값 — {ok}/{len(s)} 행에서 일치")
    if {"max_rank", "secondary_rank"} <= set(s.columns):
        ok2 = int((s["max_rank"].rank(method="dense").astype(int) == s["secondary_rank"]).sum())
        print(f"secondary_rank = max_rank 를 동점끼리 묶어 다시 매긴 순위 — {ok2}/{len(s)} 행에서 일치")

    worst = s[RANKS].idxmax(axis=1).str.replace("rank_", "", regex=False)
    print("\n각 디자인의 발목을 잡은 지표")
    for rk, wid, w in zip(s["final_rank"], s["id"], worst):
        print(f"   {int(rk):3d}위 {wid:14s} ← {w}")

위 표에서 **한 지표만 1등인 디자인은 위로 못 올라온다**는 게 보여요.
가장 나쁜 순위 하나(`max_rank`)가 그 디자인의 자리를 정하니까요. 아래 셀이 그걸 이름과 숫자로 확인합니다.

In [ ]:
if RANKS and {"design_ptm", "design_to_target_iptm"} <= set(s.columns):
    tp = s.loc[s["design_ptm"].idxmax()]
    ti = s.loc[s["design_to_target_iptm"].idxmax()]
    top = s.iloc[0]
    print(f"pTM  최고 {tp['design_ptm']:.3f}  ({tp['id']}) → 최종 {int(tp['final_rank'])}위")
    print(f"ipTM 최고 {ti['design_to_target_iptm']:.3f}  ({ti['id']}) → 최종 {int(ti['final_rank'])}위")
    if "max_rank" in s.columns:
        print(f"최종  1위 {top['id']} — 여섯 지표 중 최악이 {top['max_rank']}위로 가장 얕음")
    print("\n판정 — 한 지표만 특출난 디자인보다 여섯이 고르게 좋은 디자인이 올라와요.")
    print("실험 후보를 고를 때도 ipTM 하나로 줄세우지 말고 표의 여섯 열을 함께 보세요 (본문 5.8).")

## 8) 최종셋 밖으로 밀린 디자인 (본문 5.8)

지금까지 본 건 `budget` 개로 이미 걸러진 최종셋이에요. 선별 **전** 전체 풀에서
ipTM 이 가장 높았던 디자인이 어디로 갔는지 보면, 다양성 선택까지 포함한 선별의 성격이 드러납니다.

In [ ]:
cand = cols_in(pool, "id", "final_rank", "design_to_target_iptm", "design_ptm",
               "filter_rmsd", "designfolding-filter_rmsd", "num_filters_passed")
top3 = pool.nlargest(3, "design_to_target_iptm")[cand]
print("전체 풀에서 ipTM 상위 3")
print(top3.to_string(index=False))

inside = set(df["id"])
print(f"\n최종셋(budget {len(df)}) 포함 여부")
for _, r in top3.iterrows():
    print(f"   {r['id']:14s} {'최종셋 안' if r['id'] in inside else '최종셋 밖'}")

print("\nipTM 만 보고 골랐다면 맨 윗줄을 1순위로 뽑았을 거예요.")
print("표의 나머지 열(RMSD·통과한 필터 수)까지 보면 왜 그 디자인이 위로 못 갔는지 읽힙니다.")

## 9) 기준을 바꿔 다시 선별하기 (본문 5.9)

순위가 어떻게 만들어지는지 알았으니, 이제 기준을 내 목적에 맞게 바꿀 수 있어요.
BoltzGen 레포의 공식 노트북 `filter.ipynb` 가 하드 필터 → 품질 순위 → 다양성 선택 → 시각화를 한 번에 해 줍니다.

```python
Filter(
    design_dir=..., outdir=..., budget=5,
    use_affinity=False,            # 소분자면 True
    refolding_rmsd_threshold=2.5,  # RMSD 하드 필터
    alpha=0.1,                     # 품질 ↔ 다양성 (기본값은 작아요 — 펩타이드 0.01, 그 외 0.001)
    metrics_override={...},        # 메트릭별 중요도
    additional_filters=[...],      # 추가 하드 필터
)
```

같은 일을 CLI 로 하면 `boltzgen run ... --steps filtering` 이에요(Ch.04 7절). 필터링은 몇 초라 여러 번 돌릴 수 있습니다.
기준 하나만 조여도 후보가 얼마나 줄어드는지 지금 데이터로 세어 봅니다.

In [ ]:
THR = 2.5
if "filter_rmsd" in df.columns:
    keep = df[df["filter_rmsd"] < THR]
    print(f"refolding_rmsd_threshold={THR} → {len(keep)}/{len(df)} 생존")
    if "designfolding-filter_rmsd" in keep.columns:
        keep2 = keep[keep["designfolding-filter_rmsd"] < THR]
        print(f"additional_filters 로 'designfolding-filter_rmsd<{THR}' 까지 걸면 {len(keep2)}/{len(df)} 생존")
        print("   남는 id", ", ".join(keep2["id"].tolist()) or "없음")
    print("\n기준을 하나 더 얹었을 뿐인데 후보 수가 달라져요. 어떤 기준을 쓸지가 곧 어떤 실험을 할지예요.")
else:
    print("filter_rmsd 컬럼이 없어 이 실습은 건너뜁니다.")

### 이 챕터에서 확인한 것

메트릭 표와 2×2 그림으로 "구조는 대체로 괜찮고 인터페이스에서 갈린다"를 보고,
`rank_*` → `max_rank` → `secondary_rank` → `final_rank` 조립 과정을 데이터로 따라가
**한 지표 1등이 최종 1등이 아닌 이유**를 확인했어요. 마지막엔 기준을 2.5Å 로 조여 후보가 줄어드는 것도 봤고요.

그 "기준 바꾸기"를 명령 한 줄로 반복하는 게 실전 워크플로우입니다 —
하드 필터·가중치·다양성(`--alpha`)·길이 버킷까지 직접 만져보는 건 Ch.06 에서 이어갑니다.